In [26]:
#Add path to recording here


In [3]:
RecordingsToUse = [r'F:\Science\Freely Moving Ephys\DOI Mice\101724\G6CK10JRT',
                   r'F:\Science\Freely Moving Ephys\DOI Mice\102624\J722LT',
                   r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2025\013125\G6CK14BLN',
                   r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2025\021225',
                   r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2025\022725\G6CK14BTT',
                   r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2025\052125\G6CK14GTT',
                   r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2025\060625\J748RT']

In [30]:
RecordingPath = r'F:\Science\Freely Moving Ephys\DOI Mice\102624\J722LT'
print(RecordingPath)

F:\Science\Freely Moving Ephys\DOI Mice\102624\J722LT


In [31]:
#Importing Important Stuff
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statistics as stat 
import math 
from pylab import *
import os
import fmEphys as fme


#Defining Useful functions
def calc_std_modidx(psth,thresh):
 
    psth = psth.astype(float)
    psth = psth - np.nanmean(psth[100:750])

    std_thresh = np.nanstd(psth[100:750]*thresh)
    
    if np.nanmax((np.abs(psth[1000:1250])-std_thresh)) > 0:
        mod = 1
    else:
        mod = 0 
    return mod
    
def calc_fr_modidx(psth,thresh):

    psth = psth.astype(float)
    psth = psth - np.nanmean(psth[100:750])

    if np.nanmax(np.abs(psth[1000:1250])) > thresh:
        mod = 1
    else:
        mod = 0
    return mod

def calc_saccade_MM(psth):
    data = psth - np.nanmean(psth[50:750])
    data = data[1000:1250]
    latency = np.argmax(np.abs(data))
    MM = data[latency]
    return MM, latency

#Defining Save Paths

SavePath =  RecordingPath + "/Analysis/Gaze Shift Plots" 
if not os.path.exists(SavePath):
    os.makedirs(SavePath)

#Collecting Data Files for Analysis (Currently Just Collecting FM files and Single Unit Files)

date = os.path.basename(os.path.dirname(RecordingPath))
mouse = os.path.basename(RecordingPath)
DateAndMouse = date + '_' + mouse

#Finding Single Unit .npy file for the recording
for Items in os.listdir(RecordingPath):
    if Items.endswith("SingleUnits.npy"):
        Single_Unit_File = (RecordingPath +'/' + Items)

Single_Units = np.load(Single_Unit_File)
print(len(Single_Units),'Single Units')


#Figuring out what type of recording it is based on the number of FM blocks
FolderContent = os.listdir(RecordingPath)
count = 0
for zz in FolderContent:
    if zz.startswith("fm"):
        count = count + 1 
print(count, ' FM blocks found')


print('Loading FM Data')
#Finding FM ephys files PRE DOI and in the DARK
Pre_Dark_Data = pd.DataFrame()
Pre_Dark_Path = RecordingPath + '/fm1_dark'
Items = os.listdir(Pre_Dark_Path)
for names in Items:
    if names.endswith("ephys_props.h5"):
        Pre_Dark_File = (Pre_Dark_Path + '/' + names)
Pre_Dark_Data = pd.read_hdf(Pre_Dark_File)
Pre_Dark_Data['Mouse'] = DateAndMouse
Pre_Dark_Data = Pre_Dark_Data.loc[Single_Units]


#Finding FM ephys files PRE DOI and in the LIGHT
Pre_Light_Data = pd.DataFrame()   
Pre_Light_Path = RecordingPath + '/fm2'           
Items = os.listdir(Pre_Light_Path)
for names in Items:
    if names.endswith("ephys_props.h5"):
        Pre_Light_File = (Pre_Light_Path + '/' + names)
Pre_Light_Data = pd.read_hdf(Pre_Light_File)
Pre_Light_Data['Mouse'] = DateAndMouse
Pre_Light_Data = Pre_Light_Data.loc[Single_Units]


#Finding FM ephys files POST DOI and in the DARK
Post_Dark_Data = pd.DataFrame()   
Post_Dark_Path = RecordingPath + '/fm3_dark'
Items = os.listdir(Post_Dark_Path)
for names in Items:
    if names.endswith("ephys_props.h5"):
        Post_Dark_File = (Post_Dark_Path + '/' + names)
Post_Dark_Data = pd.read_hdf(Post_Dark_File)
Post_Dark_Data['Mouse'] = DateAndMouse
Post_Dark_Data = Post_Dark_Data.loc[Single_Units]


#Finding FM ephys files before DOI and in the light if it exists
if count == 4:
    Post_Light_Data = pd.DataFrame()   
    Post_Light_Path = RecordingPath + '/fm4'           
    Items = os.listdir(Post_Light_Path)
    for names in Items:
        if names.endswith("ephys_props.h5"):
            Post_Light_File = (Post_Light_Path + '/' + names)
    Post_Light_Data = pd.read_hdf(Post_Light_File)
    Post_Light_Data['Mouse'] = DateAndMouse
    Post_Light_Data = Post_Light_Data.loc[Single_Units]

#################################################################################################################
#Calculating and Subtracting Baseline Values from Each PSTH - Gaze Shifts

data_PreDarkLightPostDark = []
data_PostLight = []
tt = 0
 
for ii in list(Pre_Light_Data.index):

    Baselines = np.zeros([2,count])
    Mouse = Pre_Light_Data.loc[ii,'Mouse']


    #############################Pre DOI - Light Condition
    ###################Left Gaze Shifts


    R_raw_PreLight_L = Pre_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH']                                                #Get Raw Data
    R_base_PreLight_L = R_raw_PreLight_L - np.nanmean(Pre_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH'][:800])          #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PreLight_L = R_base_PreLight_L[750:1300]                                                                    #Using this window to find max modulation value
    R_max_PreLight_L = R_temp_PreLight_L[np.argmax(np.abs(R_temp_PreLight_L))]                                         #Find max modulation value (not max value b/c of negative responses)
    R_norm_PreLight_L = (R_base_PreLight_L)/np.abs(R_max_PreLight_L)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to prevent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PreLight_L,3.5)                                                                 #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PreLight_L,1)                                                                     #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                          #If thresholding not met then set responses to zero
        R_norm_PreLight_L[:] = 0
        tt = tt + 1
    
    Baselines[0,0] = np.nanmean(Pre_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH'][:800])                                #Saving Baseline value


    ###################Right Gaze Shifts
    R_raw_PreLight_R = Pre_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH']                                               #Get Raw Data
    R_base_PreLight_R = R_raw_PreLight_R - np.nanmean(Pre_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH'][:800])         #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PreLight_R = R_base_PreLight_R[750:1300]                                                                    #Using this window to find max modulation value
    R_max_PreLight_R = R_temp_PreLight_R[np.argmax(np.abs(R_temp_PreLight_R))]                                         #Find max modulation value (not max value b/c of negative responses)
    R_norm_PreLight_R = (R_base_PreLight_R)/np.abs(R_max_PreLight_R)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to prevent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PreLight_R,3.5)                                                                 #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PreLight_R,1)                                                                     #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                          #If thresholding not met then set responses to zero
        R_norm_PreLight_R[:] = 0
        tt = tt + 1

    Baselines[1,0] = np.nanmean(Pre_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH'][:800])                               #Saving Baseline value
    PreLight_GSDS = (np.abs(R_max_PreLight_L) - np.abs(R_max_PreLight_R))/(np.abs(R_max_PreLight_L) + np.abs(R_max_PreLight_R))


    #############################Pre DOI - Dark Condition
    ###################Left Gaze Shifts


    R_raw_PreDark_L = Pre_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH']                                                  #Get Raw Data
    R_base_PreDark_L = R_raw_PreDark_L - np.nanmean(Pre_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH'][:800])             #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PreDark_L = R_base_PreDark_L[750:1300]                                                                      #Using this window to find max modulation value
    R_max_PreDark_L = R_temp_PreDark_L[np.argmax(np.abs(R_temp_PreDark_L))]                                            #Find max modulation value (not max value b/c of negative responses)
    R_norm_PreDark_L = (R_base_PreDark_L)/np.abs(R_max_PreDark_L)                                                      #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to prevent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PreDark_L,3.5)                                                                  #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PreDark_L,1)                                                                      #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                          #If thresholding not met then set responses to zero
        R_norm_PreDark_L[:] = 0
        tt = tt + 1
    
    Baselines[0,1] = np.nanmean(Pre_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH'][:800])                                 #Saving Baseline value


    ###################Right Gaze Shifts
    R_raw_PreDark_R = Pre_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH']                                               #Get Raw Data
    R_base_PreDark_R = R_raw_PreDark_R - np.nanmean(Pre_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH'][:800])         #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PreDark_R = R_base_PreDark_R[750:1300]                                                                    #Using this window to find max modulation value
    R_max_PreDark_R = R_temp_PreDark_R[np.argmax(np.abs(R_temp_PreDark_R))]                                         #Find max modulation value (not max value b/c of negative responses)
    R_norm_PreDark_R = (R_base_PreDark_R)/np.abs(R_max_PreDark_R)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to prevent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PreDark_R,3.5)                                                                 #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PreDark_R,1)                                                                     #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                          #If thresholding not met then set responses to zero
        R_norm_PreDark_R[:] = 0
        tt = tt + 1

    Baselines[1,1] = np.nanmean(Pre_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH'][:800])                               #Saving Baseline value
    
    PreDark_GSDS = (np.abs(R_max_PreDark_L) - np.abs(R_max_PreDark_R))/(np.abs(R_max_PreDark_L) + np.abs(R_max_PreDark_R))


    #############################Post DOI - Dark Condition
    ###################Left Gaze Shifts


    R_raw_PostDark_L = Post_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH']                                                   #Get Raw Data
    R_base_PostDark_L = R_raw_PostDark_L - np.nanmean(Post_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH'][:800])             #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PostDark_L = R_base_PostDark_L[750:1300]                                                                       #Using this window to find max modulation value
    R_max_PostDark_L = R_temp_PostDark_L[np.argmax(np.abs(R_temp_PostDark_L))]                                            #Find max modulation value (not max value b/c of negative responses)
    R_norm_PostDark_L = (R_base_PostDark_L)/np.abs(R_max_PostDark_L)                                                      #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to Postvent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PostDark_L,3.5)                                                                  #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PostDark_L,1)                                                                      #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                           #If thresholding not met then set responses to zero
        R_norm_PostDark_L[:] = 0
        tt = tt + 1
    
    Baselines[0,2] = np.nanmean(Post_Dark_Data.loc[ii,'FmDk_gazeshift_leftPSTH'][:800])                                 #Saving Baseline value


    ###################Right Gaze Shifts
    R_raw_PostDark_R = Post_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH']                                               #Get Raw Data
    R_base_PostDark_R = R_raw_PostDark_R - np.nanmean(Post_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH'][:800])         #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
    R_temp_PostDark_R = R_base_PostDark_R[750:1300]                                                                    #Using this window to find max modulation value
    R_max_PostDark_R = R_temp_PostDark_R[np.argmax(np.abs(R_temp_PostDark_R))]                                         #Find max modulation value (not max value b/c of negative responses)
    R_norm_PostDark_R = (R_base_PostDark_R)/np.abs(R_max_PostDark_R)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to Postvent sign flip of neg responses) 

    #Thresholding Responses 
    STD_Thresh = calc_std_modidx(R_raw_PostDark_R,3.5)                                                                 #Standard Dev Threshold
    FR_Thresh = calc_fr_modidx(R_raw_PostDark_R,1)                                                                     #Spike Rate Threshold
    if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                          #If thresholding not met then set responses to zero
        R_norm_PostDark_R[:] = 0
        tt = tt + 1

    Baselines[1,2] = np.nanmean(Post_Dark_Data.loc[ii,'FmDk_gazeshift_rightPSTH'][:800])                               #Saving Baseline value
    
    PostDark_GSDS = (np.abs(R_max_PostDark_L) - np.abs(R_max_PostDark_R))/(np.abs(R_max_PostDark_L) + np.abs(R_max_PostDark_R))


    #############################Post DOI - Light Condition
    ###################Left Gaze Shifts

    if count == 4:

        R_raw_PostLight_L = Post_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH']                                                 #Get Raw Data
        R_base_PostLight_L = R_raw_PostLight_L - np.nanmean(Post_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH'][:800])          #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
        R_temp_PostLight_L = R_base_PostLight_L[750:1300]                                                                     #Using this window to find max modulation value
        R_max_PostLight_L = R_temp_PostLight_L[np.argmax(np.abs(R_temp_PostLight_L))]                                         #Find max modulation value (not max value b/c of negative responses)
        R_norm_PostLight_L = (R_base_PostLight_L)/np.abs(R_max_PostLight_L)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to Postvent sign flip of neg responses) 

        #Thresholding Responses 
        STD_Thresh = calc_std_modidx(R_raw_PostLight_L,3.5)                                                                 #Standard Dev Threshold
        FR_Thresh = calc_fr_modidx(R_raw_PostLight_L,1)                                                                     #Spike Rate Threshold
        if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                           #If thresholding not met then set responses to zero
            R_norm_PostLight_L[:] = 0
            tt = tt + 1
        
        Baselines[0,3] = np.nanmean(Post_Light_Data.loc[ii,'FmLt_gazeshift_leftPSTH'][:800])                                #Saving Baseline value


        ###################Right Gaze Shifts
        R_raw_PostLight_R = Post_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH']                                                #Get Raw Data
        R_base_PostLight_R = R_raw_PostLight_R - np.nanmean(Post_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH'][:800])         #Calculate and subtract off the baseline value (mean response from -1000:-200 before gaze onset(0))
        R_temp_PostLight_R = R_base_PostLight_R[750:1300]                                                                     #Using this window to find max modulation value
        R_max_PostLight_R = R_temp_PostLight_R[np.argmax(np.abs(R_temp_PostLight_R))]                                         #Find max modulation value (not max value b/c of negative responses)
        R_norm_PostLight_R = (R_base_PostLight_R)/np.abs(R_max_PostLight_R)                                                   #Divide the baseline subtracted response by the maximum modulation value (make sure R_max is positive to Postvent sign flip of neg responses) 

        #Thresholding Responses 
        STD_Thresh = calc_std_modidx(R_raw_PostLight_R,3.5)                                                                 #Standard Dev Threshold
        FR_Thresh = calc_fr_modidx(R_raw_PostLight_R,1)                                                                     #Spike Rate Threshold
        if (STD_Thresh == 0) or (FR_Thresh == 0):                                                                           #If thresholding not met then set responses to zero
            R_norm_PostLight_R[:] = 0
            tt = tt + 1

        Baselines[1,3] = np.nanmean(Post_Light_Data.loc[ii,'FmLt_gazeshift_rightPSTH'][:800])                                #Saving Baseline value
        PostLight_GSDS = (np.abs(R_max_PostLight_L) - np.abs(R_max_PostLight_R))/(np.abs(R_max_PostLight_L) + np.abs(R_max_PostLight_R))

        data_PostLight.append({'Mouse': Mouse, 'Cell': ii,
                    'PostLight_L': R_norm_PostLight_L, 'PostLight_L_MM': R_max_PostLight_L, 'PostLight_L_t': np.argmax(np.abs(R_norm_PostLight_L)), 'PostLight_L_Baseline': Baselines[0,3],
                    'PostLight_R': R_norm_PostLight_R, 'PostLight_R_MM': R_max_PostLight_R, 'PostLight_R_t': np.argmax(np.abs(R_norm_PostLight_R)), 'PostLight_R_Baseline': Baselines[1,3], 
                    'PostLight_L_raw': R_raw_PostLight_L, 'PostLight_R_raw': R_raw_PostLight_R, 'PostLight_GSDS': PostLight_GSDS})

        df_PostLight = pd.DataFrame(data_PostLight)


    data_PreDarkLightPostDark.append({'Mouse': Mouse, 'Cell': ii, 'Contamination_Index': Pre_Light_Data.loc[ii,'FmLt_ContamPct'],
                    'PreLight_L': R_norm_PreLight_L, 'PreLight_L_MM': R_max_PreLight_L, 'PreLight_L_t': np.argmax(np.abs(R_norm_PreLight_L)), 'PreLight_L_Baseline': Baselines[0,0],
                    'PreLight_R': R_norm_PreLight_R, 'PreLight_R_MM': R_max_PreLight_R, 'PreLight_R_t': np.argmax(np.abs(R_norm_PreLight_R)), 'PreLight_R_Baseline': Baselines[0,1], 
                    'PreLight_L_raw': R_raw_PreLight_L, 'PreLight_R_raw': R_raw_PreLight_R, 'PreLight_GSDS': PreLight_GSDS,

                    'PreDark_L': R_norm_PreDark_L, 'PreDark_L_MM': R_max_PreDark_L, 'PreDark_L_t': np.argmax(np.abs(R_norm_PreDark_L)), 'PreDark_L_Baseline': Baselines[0,1],
                    'PreDark_R': R_norm_PreDark_R, 'PreDark_R_MM': R_max_PreDark_R, 'PreDark_R_t': np.argmax(np.abs(R_norm_PreDark_R)), 'PreDark_R_Baseline': Baselines[1,1],
                    'PreDark_L_raw': R_raw_PreDark_L, 'PreDark_R_raw': R_raw_PreDark_R, 'PreDark_GSDS': PreDark_GSDS,

                    'PostDark_L': R_norm_PostDark_L, 'PostDark_L_MM': R_max_PostDark_L, 'PostDark_L_t': np.argmax(np.abs(R_norm_PostDark_L)), 'PostDark_L_Baseline': Baselines[0,2],
                    'PostDark_R': R_norm_PostDark_R, 'PostDark_R_MM': R_max_PostDark_R, 'PostDark_R_t': np.argmax(np.abs(R_norm_PostDark_R)), 'PostDark_R_Baseline': Baselines[1,2],
                    'PostDark_L_raw': R_raw_PostDark_L, 'PostDark_R_raw': R_raw_PostDark_R, 'PostDark_GSDS': PostDark_GSDS})                                                   

df = pd.DataFrame(data_PreDarkLightPostDark)

#################################################################################################################
#Selecting the preferred Gaze Shift Response before and after DOI for future analysis 

Pref_Gaze_PreLight = []
Pref_Gaze_PreLight_Raw = []
Dir_Pref_PreLight = []
Pref_Gaze_EventT_PreLight = []

NonPref_Gaze_PreLight = []
NonPref_Gaze_PreLight_Raw = []

Pref_Gaze_PreDark = []
Pref_Gaze_PreDark_Raw = []
Pref_Gaze_EventT_PreDark = []

NonPref_Gaze_PreDark = []
NonPref_Gaze_PreDark_Raw = []

Pref_Gaze_PostDark = []
Pref_Gaze_PostDark_Raw = []
Pref_Gaze_EventT_PostDark = []

NonPref_Gaze_PostDark = []
NonPref_Gaze_PostDark_Raw = []

#PreLight_SpikeT = []
#PreDark_SpikeT = []
#PostDark_SpikeT = []

if count == 4:
    Pref_Gaze_PostLight = []
    Pref_Gaze_PostLight_Raw = []
    Pref_Gaze_EventT_PostLight = []
    #PostLight_SpikeT = []



for ii in list(range(0,len(df))):
    if df.loc[ii,'PreLight_GSDS'] > 0:

        Dir_Pref_PreLight.append('L')
        Pref_Gaze_PreLight.append(df.loc[ii,'PreLight_L'])
        Pref_Gaze_PreLight_Raw.append(df.loc[ii,'PreLight_L_raw'])
        NonPref_Gaze_PreLight.append(df.loc[ii,'PreLight_R'])
        NonPref_Gaze_PreLight_Raw.append(df.loc[ii,'PreLight_R_raw'])

        Pref_Gaze_PreDark.append(df.loc[ii,'PreDark_L'])
        Pref_Gaze_PreDark_Raw.append(df.loc[ii,'PreDark_L_raw'])
        NonPref_Gaze_PreDark.append(df.loc[ii,'PreDark_R'])
        NonPref_Gaze_PreDark_Raw.append(df.loc[ii,'PreDark_R_raw'])

        Pref_Gaze_PostDark.append(df.loc[ii,'PostDark_L'])
        Pref_Gaze_PostDark_Raw.append(df.loc[ii,'PostDark_L_raw'])
        NonPref_Gaze_PostDark.append(df.loc[ii,'PostDark_R'])
        NonPref_Gaze_PostDark_Raw.append(df.loc[ii,'PostDark_R_raw'])

        if count == 4:
            Pref_Gaze_PostLight.append(df_PostLight.loc[ii,'PostLight_L'])
            Pref_Gaze_PostLight_Raw.append(df_PostLight.loc[ii,'PostLight_L_raw'])

    else:

        Dir_Pref_PreLight.append('R')
        Pref_Gaze_PreLight.append(df.loc[ii,'PreLight_R'])
        Pref_Gaze_PreLight_Raw.append(df.loc[ii,'PreLight_R_raw'])
        NonPref_Gaze_PreLight.append(df.loc[ii,'PreLight_L'])
        NonPref_Gaze_PreLight_Raw.append(df.loc[ii,'PreLight_L_raw'])

        Pref_Gaze_PreDark.append(df.loc[ii,'PreDark_R'])
        Pref_Gaze_PreDark_Raw.append(df.loc[ii,'PreDark_R_raw'])      
        NonPref_Gaze_PreDark.append(df.loc[ii,'PreDark_L'])
        NonPref_Gaze_PreDark_Raw.append(df.loc[ii,'PreDark_L_raw'])

        Pref_Gaze_PostDark.append(df.loc[ii,'PostDark_R'])
        Pref_Gaze_PostDark_Raw.append(df.loc[ii,'PostDark_R_raw'])
        NonPref_Gaze_PostDark.append(df.loc[ii,'PostDark_L'])
        NonPref_Gaze_PostDark_Raw.append(df.loc[ii,'PostDark_L_raw'])

        if count == 4:
            Pref_Gaze_PostLight.append(df_PostLight.loc[ii,'PostLight_R'])
            Pref_Gaze_PostLight_Raw.append(df_PostLight.loc[ii,'PostLight_R_raw'])

    
#Adding the Pref_Gaze Shift (Pre and Post) to the dataframe
df['Dir_Pref_PreLight'] = Dir_Pref_PreLight
df['Pref_Gaze_PreLight'] = Pref_Gaze_PreLight
df['Pref_Gaze_PreLight_Raw'] = Pref_Gaze_PreLight_Raw
df['Pref_Gaze_PreDark'] = Pref_Gaze_PreDark
df['Pref_Gaze_PreDark_Raw'] = Pref_Gaze_PreDark_Raw
df['Pref_Gaze_PostDark'] = Pref_Gaze_PostDark
df['Pref_Gaze_PostDark_Raw'] = Pref_Gaze_PostDark_Raw

df['NonPref_Gaze_PreLight'] = NonPref_Gaze_PreLight
df['NonPref_Gaze_PreLight_Raw'] = NonPref_Gaze_PreLight_Raw
df['NonPref_Gaze_PreDark'] = NonPref_Gaze_PreDark
df['NonPref_Gaze_PreDark_Raw'] = NonPref_Gaze_PreDark_Raw
df['NonPref_Gaze_PostDark'] = NonPref_Gaze_PostDark
df['NonPref_Gaze_PostDark_Raw'] = NonPref_Gaze_PostDark_Raw

if count == 4:
    df_PostLight['Pref_Gaze_PostLight'] = Pref_Gaze_PostLight
    df_PostLight['Pref_Gaze_PostLight_Raw'] = Pref_Gaze_PostLight_Raw


print('Saving Gaze Shift Response Data')
df.to_hdf(os.path.join(os.path.join(RecordingPath,'GazeData_PreDLPostD_PreDOI.h5')), 'w')
if count == 4:
    df_PostLight.to_hdf(os.path.join(os.path.join(RecordingPath,'GazeData_PostLight.h5')), 'w')


SavePath =  RecordingPath + "/Analysis/Light-Dark"
if not os.path.exists(SavePath):
    os.makedirs(SavePath)


SavePath_SingleCells = SavePath + "/Single Units"
if not os.path.exists(SavePath_SingleCells):
    os.makedirs(SavePath_SingleCells)


#Making Figs and Saving
#################################################################################################################
#Baseline Levels

PreLight_Baseline = np.nanmean(np.stack(df['Pref_Gaze_PreLight_Raw'])[:,0:800],axis=1)
PreDark_Baseline = np.nanmean(np.stack(df['Pref_Gaze_PreDark_Raw'])[:,0:800],axis=1)
PostDark_Baseline = np.nanmean(np.stack(df['Pref_Gaze_PostDark_Raw'])[:,0:800],axis=1)

if count == 4:
    PostLight_Baseline = np.nanmean(np.stack(df_PostLight['Pref_Gaze_PostLight_Raw'])[:,0:800],axis=1)
    Labels = ['Pre-Light','Pre-Dark','Post-Dark','Post-Light']
    Baseline_STE =[(np.nanstd(PreLight_Baseline)/sqrt(len(PreLight_Baseline))),(np.nanstd(PreDark_Baseline)/sqrt(len(PreDark_Baseline))),(np.nanstd(PostDark_Baseline)/sqrt(len(PostDark_Baseline))),(np.nanstd(PostLight_Baseline)/sqrt(len(PostLight_Baseline)))]
else:
    Labels = ['Pre-Light','Pre-Dark','Post-Dark']
    Baseline_STE =[(np.nanstd(PreLight_Baseline)/sqrt(len(PreLight_Baseline))),(np.nanstd(PreDark_Baseline)/sqrt(len(PreDark_Baseline))),(np.nanstd(PostDark_Baseline)/sqrt(len(PostDark_Baseline)))]

plt.figure(figsize=(10,10))
plt.bar(1,np.nanmean(PreLight_Baseline),color='grey')
plt.bar(2,np.nanmean(PreDark_Baseline),color='grey',hatch='o')
plt.bar(3,np.nanmean(PostDark_Baseline),color='coral',hatch='o')

if count  == 4:
    plt.bar(4,np.nanmean(PostLight_Baseline),color='coral')
    plt.errorbar(4,np.nanmean(PostLight_Baseline),Baseline_STE[3],color='black')

plt.errorbar(1,np.nanmean(PreLight_Baseline),Baseline_STE[0],color='black')
plt.errorbar(2,np.nanmean(PreDark_Baseline),Baseline_STE[1],color='black')
plt.errorbar(3,np.nanmean(PostDark_Baseline),Baseline_STE[2],color='black')

plt.title('Baseline Firing Rates During Free Movement')
plt.legend(Labels)
plt.ylabel('Spikes/sec')

plt.savefig(os.path.join(SavePath,'BaselineFR.png'),bbox_inches='tight')
plt.close()

#################################################################################################################
#Plotting Individual Gaze Responses Pre, Post, Light, Dark

for ii in range(0,len(df)):
    PreLight_Response = df.loc[ii,'Pref_Gaze_PreLight']
    PreDark_Response = df.loc[ii,'Pref_Gaze_PreDark']
    PostDark_Response = df.loc[ii,'Pref_Gaze_PostDark']

    plt.figure(figsize=(10,10))
    plt.plot(PreLight_Response, color = 'gold',label = 'Pre-Light')
    plt.plot(PreDark_Response, color = 'black',label = 'Pre-Dark')
    plt.plot(PostDark_Response, color = 'darkred',label = 'Post-Dark')

    if count == 4:
        PostLight_Response = df_PostLight.loc[ii,'Pref_Gaze_PostLight']
        plt.plot(PostLight_Response, color = 'coral',label = 'Post-Light')
    plt.legend()

    plt.savefig(os.path.join(SavePath_SingleCells,'Single_Unit_'+ str(ii)+'.png'),bbox_inches='tight')
    plt.close()

#################################################################################################################
#Plotting Individual Gaze Responses Pre, Post, Light, Dark

plt.figure(figsize=(10,10))
plt.plot(np.nanmean(np.stack(df.loc[:,'Pref_Gaze_PreLight']),axis=0),color = 'gold',label='Pre-Light')
plt.plot(np.nanmean(np.stack(df.loc[:,'Pref_Gaze_PreDark']),axis=0),color = 'black',label = 'Pre-Dark')
plt.plot(np.nanmean(np.stack(df.loc[:,'Pref_Gaze_PostDark']),axis=0),color = 'darkred',label = 'Post-Dark')
if count == 4:
    plt.plot(np.nanmean(np.stack(df_PostLight.loc[:,'Pref_Gaze_PostLight']),axis=0),color = 'coral',label = 'Post-Light')
plt.legend()

plt.savefig(os.path.join(SavePath,'Average Gaze Responses.png'),bbox_inches='tight')
plt.close()
print('Done!')

47 Single Units
3  FM blocks found
Loading FM Data
Saving Gaze Shift Response Data
Done!


In [15]:
Pre_Light_Data.columns

Index(['FmLt_Amplitude', 'FmLt_ContamPct', 'FmLt_KSLabel', 'FmLt_amp',
       'FmLt_ch', 'FmLt_depth', 'FmLt_fr', 'FmLt_group', 'FmLt_n_spikes',
       'FmLt_sh', 'FmLt_waveform', 'FmLt_spikeT', 'FmLt_t0', 'FmLt_spikeTraw',
       'FmLt_rate', 'FmLt_contrast_tuning_bins', 'FmLt_contrast_tuning',
       'FmLt_contrast_tuning_err', 'FmLt_spike_triggered_average',
       'FmLt_spike_triggered_variance', 'FmLt_saccade_rightT',
       'FmLt_saccade_leftT', 'FmLt_compensatory_rightT',
       'FmLt_compensatory_leftT', 'FmLt_gazeshift_leftT',
       'FmLt_gazeshift_rightT', 'FmLt_saccade_rightPSTH',
       'FmLt_saccade_leftPSTH', 'FmLt_compensatory_rightPSTH',
       'FmLt_compensatory_leftPSTH', 'FmLt_gazeshift_leftPSTH',
       'FmLt_gazeshift_rightPSTH', 'FmLt_pupilradius_tuning_bins',
       'FmLt_pupilradius_tuning', 'FmLt_pupilradius_tuning_err',
       'FmLt_theta_tuning_bins', 'FmLt_theta_tuning', 'FmLt_theta_tuning_err',
       'FmLt_phi_tuning_bins', 'FmLt_phi_tuning', 'FmLt_phi_tu